<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module34a/Lab06.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lab 6 — Spin Orbitals, Fermion-to-Qubit Encodings, and UCCSD

**Maps to:** Module 4, Lessons 1–3 (Atomic/Molecular/Spin Orbitals; CI → CC → UCC → UCCSD;
Configuration Mix in H$_2$)

**Time:** ~60 minutes (instructor walkthrough ~15 min)

---

### The question this lab answers

Lab 5 solved H$_2$ with **two** qubits and **one** parameter. That was a gift from
symmetry. This lab shows the honest picture — four spin orbitals, four qubits, the full
UCCSD ansatz — so you can see what the reduction bought, and so the machinery transfers
to molecules where no such shortcut exists.

### After this lab you can
1. Map spin orbitals to qubits and write the Hartree–Fock reference as a bitstring.
2. Explain the $Z$ strings that Jordan–Wigner sprinkles into every excitation operator.
3. Build HF + UCCSD circuits with Qiskit Nature and count their parameters.
4. Compare Jordan–Wigner, Bravyi–Kitaev, and parity encodings on qubit count, Pauli-term
   count, and **measurement settings**.
5. Run VQE in the 4-qubit encoding and confirm it reproduces the Lab 5 energy.

In [ ]:
# %pip install -q qiskit qiskit-aer qiskit-nature pyscf matplotlib scipy
import numpy as np, time
from scipy.optimize import minimize
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, SparsePauliOp, Operator
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2

from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import (JordanWignerMapper, BravyiKitaevMapper,
                                            ParityMapper)
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD

np.set_printoptions(precision=4, suppress=True)
HARTREE_TO_EV = 27.2114

problem = PySCFDriver(atom="H 0 0 0; H 0 0 0.735", basis="sto3g",
                      unit=DistanceUnit.ANGSTROM).run()
print("spatial orbitals :", problem.num_spatial_orbitals)
print("spin orbitals    :", 2*problem.num_spatial_orbitals)
print("electrons (a, b) :", problem.num_particles)
print("nuclear repulsion:", round(problem.nuclear_repulsion_energy, 6), "Ha")

## 1. Four "lanes": the spin orbitals

Minimal-basis H$_2$ has two **molecular orbitals** — bonding $\sigma_g$ and antibonding
$\sigma_u^*$ — and each holds two spins. Four spin orbitals, i.e. four lanes for two cars:

| index | orbital | lecture label |
|---|---|---|
| 0 | bonding, ↑ | $\sigma_g(\alpha)$ |
| 1 | bonding, ↓ | $\sigma_g(\beta)$ |
| 2 | antibonding, ↑ | $\sigma_u^*(\alpha)$ |
| 3 | antibonding, ↓ | $\sigma_u^*(\beta)$ |

One qubit per spin orbital: $|1\rangle$ = occupied, $|0\rangle$ = empty. Hartree–Fock puts
both electrons in the bonding orbital: $|1100\rangle$ in the lecture's ordering.

> ### ⚠️ Two orderings to keep straight
> The slides order spin orbitals **by orbital**: $(\sigma_g\uparrow, \sigma_g\downarrow,
> \sigma_u^*\uparrow, \sigma_u^*\downarrow)$, giving HF $=|1100\rangle$.
> Qiskit Nature orders them **by spin block**: all $\alpha$ first, then all $\beta$, i.e.
> $(\sigma_g\uparrow, \sigma_u^*\uparrow, \sigma_g\downarrow, \sigma_u^*\downarrow)$,
> giving HF $=|1010\rangle$. Same physics, different labels on the wires. Print the
> circuit and check which qubits get an `X` — that always settles it.

In [ ]:
hf4 = HartreeFock(problem.num_spatial_orbitals, problem.num_particles, JordanWignerMapper())
print(hf4.decompose().draw(output="text"))
print("\nHartree-Fock state (Qiskit prints q3q2q1q0):",
      Statevector(hf4).probabilities_dict())
print("X gates on qubits 0 and 2  ->  alpha-bonding and beta-bonding are occupied.")

## 2. Excitations = moving a car between lanes

An excitation operator $a_p^\dagger a_q$ removes an electron from lane $q$ and puts it in
lane $p$. In UCC we always use the **anti-Hermitian** combination
$a_p^\dagger a_q - a_q^\dagger a_p$, because $e^{(\text{anti-Hermitian})}$ is guaranteed
unitary — the reason your slide writes $e^{T - T^\dagger}$ rather than $e^{T}$.

### Exercise 1 — see the Jordan–Wigner $Z$ string appear

Map the single excitation $a_2^\dagger a_0 - a_0^\dagger a_2$ to Pauli operators.

In [ ]:
single = FermionicOp({"+_2 -_0": 1.0, "+_0 -_2": -1.0}, num_spin_orbitals=4)
P_single = JordanWignerMapper().map(single)
print(P_single)
print("\nlabels are printed q3 q2 q1 q0:")
for pauli, c in zip(P_single.paulis, P_single.coeffs):
    print(f"   {pauli.to_label()}   coeff {c}")

Read it: the operator is $\tfrac{i}{2}\,(X_2 Z_1 Y_0 - Y_2 Z_1 X_0)$.

* $X$ and $Y$ on qubits **2** and **0** — the two lanes actually involved. Same
  $X\!\cdot\!Y - Y\!\cdot\!X$ pattern as the Givens rotation in Labs 3 and 5, which is why
  it produces **real** amplitudes.
* A $Z$ on qubit **1**, which is not involved in the move at all. That $Z$ is pure
  bookkeeping: electrons are fermions, so swapping two of them flips the sign of the
  wavefunction, and Jordan–Wigner pays for that with a string of $Z$'s spanning every
  orbital between $p$ and $q$.

The cost is real. A $Z$ string of length $\ell$ turns into $\ell$ extra CNOTs, and $\ell$
grows with the molecule under Jordan–Wigner. Bravyi–Kitaev exists to make it grow like
$\log$ instead.

### Exercise 2 — build the circuit for this excitation, and check it

Any $e^{-i\frac{\theta}{2}P}$ for a Pauli string $P$ is compiled the same way:
rotate each qubit into the $Z$ basis, compute the parity of all involved qubits with a
CNOT ladder, apply one $R_z$, then undo. This is the Lab 2 recipe with more wires.

In [ ]:
def pauli_evolution(pauli_label, theta, n=4):
    '''Circuit for exp(-i theta/2 * P), P given as a Qiskit label (q_{n-1}...q_0).'''
    qc = QuantumCircuit(n)
    active = [i for i, ch in enumerate(reversed(pauli_label)) if ch != "I"]
    for i in active:                      # rotate into Z basis
        ch = pauli_label[::-1][i]
        if ch == "X": qc.h(i)
        elif ch == "Y": qc.sdg(i); qc.h(i)
    # TODO: CNOT ladder from active[0] up to active[-1]
    ...
    # TODO: one rz(theta) on active[-1]
    ...
    # TODO: undo the ladder (reverse order)
    ...
    for i in active:                      # undo the basis change
        ch = pauli_label[::-1][i]
        if ch == "X": qc.h(i)
        elif ch == "Y": qc.h(i); qc.s(i)
    return qc

theta = 0.4
qc = pauli_evolution("IXZY", theta)
print(qc.draw(output="text"))

import scipy.linalg as la
target = la.expm(-1j*theta/2 * SparsePauliOp("IXZY").to_matrix())
assert np.allclose(Operator(qc).data, target)
print("\nPASS")

## 3. UCCSD: all singles and all doubles

$$|\Psi_{\text{UCCSD}}(\vec\theta)\rangle = e^{\,T-T^\dagger}\,|\Phi_{\text{HF}}\rangle,
\qquad T = T_1 + T_2 .$$

For H$_2$ in this basis there are exactly **three** independent excitation amplitudes:
two singles ($0\!\to\!2$ and $1\!\to\!3$, i.e. one electron promoted) and one double
(both promoted). Those are the $U_{1a}, U_{1b}, U_2$ blocks on your ansatz slide.

In [ ]:
mapper_jw = JordanWignerMapper()
H_jw = mapper_jw.map(problem.hamiltonian.second_q_op())
hf_jw = HartreeFock(problem.num_spatial_orbitals, problem.num_particles, mapper_jw)
ansatz_jw = UCCSD(problem.num_spatial_orbitals, problem.num_particles, mapper_jw,
                  initial_state=hf_jw)

print("qubits            :", ansatz_jw.num_qubits)
print("variational params:", ansatz_jw.num_parameters)
print("excitations       :")
for exc in ansatz_jw.excitation_list:
    kind = "single" if len(exc[0]) == 1 else "double"
    print(f"   {kind}: occupied {exc[0]} -> virtual {exc[1]}")

### Exercise 3 — decompose and count the real cost

In [ ]:
sim = AerSimulator()
isa_jw = transpile(ansatz_jw, sim, optimization_level=1)
print("UCCSD (4 qubits) after transpiling:")
print("   depth     :", isa_jw.depth())
print("   CNOTs     :", isa_jw.count_ops().get("cx", 0))
print("   total gates:", sum(isa_jw.count_ops().values()))

## 4. Run VQE in the 4-qubit encoding

Same loop as Lab 5, three parameters instead of one.

In [ ]:
estimator = EstimatorV2(options={"default_precision": 0.0})

def vqe(ansatz, H, E_nuc, x0=None, maxiter=400):
    x0 = np.zeros(ansatz.num_parameters) if x0 is None else x0
    calls = {"n": 0}
    def f(x):
        calls["n"] += 1
        return float(estimator.run([(ansatz, H, x)]).result()[0].data.evs)
    res = minimize(f, x0, method="COBYLA", options={"maxiter": maxiter})
    return res.x, res.fun + E_nuc, calls["n"]

t0 = time.time()
x_opt, E_jw, n_calls = vqe(isa_jw, H_jw, problem.nuclear_repulsion_energy)
E_exact = np.linalg.eigvalsh(H_jw.to_matrix())[0] + problem.nuclear_repulsion_energy

print(f"VQE (JW, 4 qubits): E = {E_jw:.8f} Ha  ({n_calls} evaluations, {time.time()-t0:.1f} s)")
print(f"exact             : E = {E_exact:.8f} Ha")
print(f"error             : {1000*abs(E_jw-E_exact):.6f} mHa")
print(f"optimal amplitudes: {np.round(x_opt, 5)}")
assert abs(E_jw - E_exact) < 1e-6
print("\nPASS -- same -1.137306 Ha as the 2-qubit Lab 5 run.")
print("Note the two single-excitation amplitudes are ~0: in H2 the singles do nothing.")

## 5. Encodings side by side

Three ways to turn fermions into qubits, all exact, all giving the same energy:

* **Jordan–Wigner** — one qubit per spin orbital, occupation stored directly, long $Z$
  strings.
* **Bravyi–Kitaev** — a mix of occupation and *parity* information, so both occupancy and
  parity lookups cost $O(\log n)$; shorter Pauli strings.
* **Parity + two-qubit reduction** — stores parities, then exploits the fact that total
  particle number and spin are conserved to **delete two qubits**. This is where Lab 5's
  two-qubit model came from.

### Exercise 4 — build the comparison table

Fill in the loop and report, for each mapper: qubits, Pauli terms, measurement settings
(qubit-wise commuting groups), and ground-state energy.

In [ ]:
rows = []
mappers = [("Jordan-Wigner", JordanWignerMapper()),
           ("Bravyi-Kitaev", BravyiKitaevMapper()),
           ("Parity", ParityMapper()),
           ("Parity + 2q reduction", ParityMapper(num_particles=problem.num_particles))]

for name, m in mappers:
    Hm = m.map(problem.hamiltonian.second_q_op())
    # TODO: groups = Hm.paulis.group_qubit_wise_commuting()
    # TODO: e = lowest eigenvalue of Hm.to_matrix() plus the nuclear repulsion
    ...
    rows.append((name, Hm.num_qubits, len(Hm), len(groups), e))

print(f"{'mapper':<24}{'qubits':>7}{'terms':>7}{'settings':>10}{'E0 (Ha)':>14}")
for r in rows:
    print(f"{r[0]:<24}{r[1]:>7}{r[2]:>7}{r[3]:>10}{r[4]:>14.6f}")
assert all(np.isclose(r[4], rows[0][4], atol=1e-8) for r in rows)
print("\nPASS")

Note the **measurement settings** column: Jordan–Wigner needs 5 experiments per energy
evaluation, Bravyi–Kitaev only 2. Same 15 terms, less than half the QPU time. That column
is often more important than the qubit count.

### Exercise 5 — what the reduction actually bought

Build the 2-qubit UCCSD ansatz and put the two runs side by side.

In [ ]:
mapper_2q = ParityMapper(num_particles=problem.num_particles)
H_2q = mapper_2q.map(problem.hamiltonian.second_q_op())
hf_2q = HartreeFock(problem.num_spatial_orbitals, problem.num_particles, mapper_2q)
ansatz_2q = UCCSD(problem.num_spatial_orbitals, problem.num_particles, mapper_2q,
                  initial_state=hf_2q)
isa_2q = transpile(ansatz_2q, sim, optimization_level=1)

t0 = time.time(); _, E_2q, calls_2q = vqe(isa_2q, H_2q, problem.nuclear_repulsion_energy)
t_2q = time.time() - t0

print(f"{'':<14}{'qubits':>7}{'terms':>7}{'params':>8}{'depth':>7}{'CNOTs':>7}{'E (Ha)':>13}")
print(f"{'JW 4-qubit':<14}{4:>7}{len(H_jw):>7}{isa_jw.num_parameters:>8}"
      f"{isa_jw.depth():>7}{isa_jw.count_ops().get('cx',0):>7}{E_jw:>13.6f}")
print(f"{'Reduced 2q':<14}{2:>7}{len(H_2q):>7}{isa_2q.num_parameters:>8}"
      f"{isa_2q.depth():>7}{isa_2q.count_ops().get('cx',0):>7}{E_2q:>13.6f}")

print(f"\nCNOT reduction: {isa_jw.count_ops().get('cx',0) / max(isa_2q.count_ops().get('cx',1),1):.1f}x")
print("Identical energy, a fraction of the circuit. On today's hardware that ratio")
print("is the difference between a usable result and noise.")

## 6. Why this does not scale (yet)

The reduction we used relies on H$_2$'s particular symmetries. The general trend:

| molecule | spin orbitals (STO-3G) | qubits (JW) | UCCSD parameters |
|---|---|---|---|
| H$_2$ | 4 | 4 | 3 |
| LiH | 12 | 12 | ~90 |
| H$_2$O | 14 | 14 | ~140 |
| N$_2$ | 20 | 20 | ~500 |

### Exercise 6 (optional, ~2 min runtime) — try LiH

If PySCF is installed, run the cell below. Do **not** run VQE on it — just look at the
size. It is the honest scale of the problem.

In [ ]:
lih = PySCFDriver(atom="Li 0 0 0; H 0 0 1.6", basis="sto3g",
                  unit=DistanceUnit.ANGSTROM).run()
H_lih = JordanWignerMapper().map(lih.hamiltonian.second_q_op())
hf_lih = HartreeFock(lih.num_spatial_orbitals, lih.num_particles, JordanWignerMapper())
anz_lih = UCCSD(lih.num_spatial_orbitals, lih.num_particles, JordanWignerMapper(),
                initial_state=hf_lih)
print("LiH:  qubits =", H_lih.num_qubits,
      " Pauli terms =", len(H_lih),
      " measurement settings =", len(H_lih.paulis.group_qubit_wise_commuting()),
      " UCCSD parameters =", anz_lih.num_parameters)
print("\nCompare H2: 4 qubits, 15 terms, 5 settings, 3 parameters.")

## Checkpoint

1. Why does UCC exponentiate $T - T^\dagger$ rather than $T$?
2. What is the $Z_1$ in $X_2 Z_1 Y_0$ doing physically? What would happen if you dropped it?
3. In the H$_2$ UCCSD solution, the single-excitation amplitudes came out near zero.
   Why? (Hint: which excitations change the total spin?)
4. Jordan–Wigner and Bravyi–Kitaev both use 4 qubits and 15 terms for H$_2$, but BK needs
   2 measurement settings instead of 5. Where does the saving come from?
5. LiH needs ~90 UCCSD parameters. If COBYLA needs roughly $10\times$ the number of
   parameters in energy evaluations, and each evaluation costs 12 settings × 4000 shots,
   estimate the total shot count for a single geometry.

### What is next
**Lab 7** takes the H$_2$ circuit off the simulator and runs it on a real IBM Quantum
processor — with, and without, error mitigation.